In [1]:
import pandas as pd, numpy as np, joblib, os
from collections import defaultdict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error
os.makedirs("models", exist_ok=True)

games = pd.read_csv("data/games_clean.csv", parse_dates=["Date"])
games = games.sort_values(["Date", "GameKey"]).reset_index(drop=True)

In [2]:
K, HFA, START = 30, 40, 1500
elo = defaultdict(lambda: START)
hist = defaultdict(list)     # team -> [(result, margin), ...]
last_date = {}

def form(team):
    h = hist[team][-5:]
    if not h: return 0.5, 0.0
    return np.mean([x[0] for x in h]), np.mean([x[1] for x in h])

def rest(team, date):
    return min((date - last_date[team]).days, 21) if team in last_date else 14

feat, season = [], None
for r in games.itertuples():
    if r.Year != season:                       # pull ratings toward average each new season
        for t in list(elo): elo[t] = 0.75 * elo[t] + 0.25 * START
        season = r.Year
    h, a = r.HomeTeam, r.AwayTeam
    hf, hm = form(h); af, am = form(a)
    feat.append(dict(EloDiff=elo[h] - elo[a], FormDiff=hf - af, MarginDiff=hm - am,
                     RestDiff=rest(h, r.Date) - rest(a, r.Date)))
    # update AFTER recording the features
    s = 1.0 if r.Margin > 0 else 0.0 if r.Margin < 0 else 0.5
    exp = 1 / (1 + 10 ** (-(elo[h] + HFA - elo[a]) / 400))
    elo[h] += K * (s - exp); elo[a] -= K * (s - exp)
    hist[h].append((s, r.Margin)); hist[a].append((1 - s, -r.Margin))
    last_date[h] = last_date[a] = r.Date

games = pd.concat([games, pd.DataFrame(feat)], axis=1)
print(pd.DataFrame(sorted(elo.items(), key=lambda x: -x[1]), columns=["Team", "Elo"]).round(0).head(18))

                      Team     Elo
0                 Brisbane  1693.0
1                  Geelong  1649.0
2                 Hawthorn  1614.0
3   Greater Western Sydney  1606.0
4              Collingwood  1605.0
5                 Adelaide  1592.0
6                Fremantle  1591.0
7         Western Bulldogs  1571.0
8                   Sydney  1562.0
9               Gold Coast  1557.0
10           Port Adelaide  1485.0
11                 Carlton  1458.0
12                St Kilda  1455.0
13               Melbourne  1399.0
14                Essendon  1354.0
15         North Melbourne  1301.0
16                Richmond  1291.0
17              West Coast  1216.0


In [3]:
NUM = ["EloDiff", "FormDiff", "MarginDiff", "RestDiff", "IsFinal"]
CAT = ["VenueGroup"]
data = games[games["Margin"] != 0]
train, test = data[data.Year <= 2022], data[data.Year >= 2023]
print("Train games:", len(train), "| Test games:", len(test))

def build(clf, num=NUM):
    pre = ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)])
    return Pipeline([("pre", pre), ("clf", clf)])

models = {
    "Elo only (logistic)": build(LogisticRegression(max_iter=1000), ["EloDiff"]),
    "All features (logistic)": build(LogisticRegression(max_iter=1000)),
    "All features (grad. boosting)": build(HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150)),
}
print(f"Always pick home team: accuracy {test['HomeWin'].mean():.3f}")
for name, mdl in models.items():
    mdl.fit(train, train["HomeWin"])
    p = mdl.predict_proba(test)[:, 1]
    print(f"{name:32s} accuracy {accuracy_score(test['HomeWin'], p > .5):.3f} | log loss {log_loss(test['HomeWin'], p):.3f}")

Train games: 2216 | Test games: 642
Always pick home team: accuracy 0.572
Elo only (logistic)              accuracy 0.670 | log loss 0.589
All features (logistic)          accuracy 0.670 | log loss 0.587
All features (grad. boosting)    accuracy 0.678 | log loss 0.596


In [4]:
match_model = build(LogisticRegression(max_iter=1000))
match_model.fit(data, data["HomeWin"])

state = {"model": match_model, "elo": dict(elo),
         "form": {t: form(t) for t in elo},
         "venues": sorted(games["VenueGroup"].unique())}
joblib.dump(state, "models/match_model.joblib")

def predict_match(home, away, venue="MCG", is_final=0):
    if home not in state["elo"] or away not in state["elo"]:
        raise ValueError("Unknown team")
    hf, hm = state["form"][home]; af, am = state["form"][away]
    X = pd.DataFrame([dict(EloDiff=state["elo"][home] - state["elo"][away],
                           FormDiff=hf - af, MarginDiff=hm - am, RestDiff=0, IsFinal=is_final,
                           VenueGroup=venue if venue in state["venues"] else "Other")])
    p = match_model.predict_proba(X)[0, 1]
    return {"home": home, "away": away, "venue": venue,
            "home_win_prob": round(p, 3), "winner": home if p >= .5 else away}

print(predict_match("Collingwood", "Carlton", "MCG"))
print(predict_match("Geelong", "Adelaide", "Kardinia Park"))

{'home': 'Collingwood', 'away': 'Carlton', 'venue': 'MCG', 'home_win_prob': np.float64(0.691), 'winner': 'Collingwood'}
{'home': 'Geelong', 'away': 'Adelaide', 'venue': 'Kardinia Park', 'home_win_prob': np.float64(0.821), 'winner': 'Geelong'}


In [5]:
stats = pd.read_csv("data/stats_clean.csv")
players = pd.read_csv("data/players_clean.csv", parse_dates=["Dob"])

s = (stats.merge(games[["GameKey", "Date", "HomeTeam"]], on="GameKey")
          .merge(players[["PlayerId", "Height", "Weight", "Dob", "Position"]], on="PlayerId", how="left"))
s["IsHome"] = (s["Team"] == s["HomeTeam"]).astype(int)
s["Age"] = (s["Date"] - s["Dob"]).dt.days / 365.25
s["Position"] = s["Position"].fillna("Unknown")
s = s.sort_values(["PlayerId", "Date"]).reset_index(drop=True)

g = s.groupby("PlayerId")
for c in ["Fantasy", "Goals", "Disposals", "PctPlayed"]:
    s[f"{c}_L5"] = g[c].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
for c in ["Fantasy", "Goals", "Disposals"]:
    s[f"{c}_L20"] = g[c].transform(lambda x: x.shift(1).rolling(20, min_periods=1).mean())
    s[f"{c}_Career"] = g[c].transform(lambda x: x.shift(1).expanding().mean())
s["PrevGames"] = g.cumcount()
s = s[s["PrevGames"] > 0].copy()          # drop debut games (no history)

FEATS = ["Fantasy_L5", "Fantasy_L20", "Fantasy_Career", "Goals_L5", "Goals_L20", "Goals_Career",
         "Disposals_L5", "Disposals_L20", "Disposals_Career", "PctPlayed_L5",
         "PrevGames", "IsHome", "Age", "Height", "Weight", "Position"]

def make_reg():
    pre = ColumnTransformer([("pos", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["Position"])],
                            remainder="passthrough")
    return Pipeline([("pre", pre), ("reg", HistGradientBoostingRegressor(max_depth=4, learning_rate=0.05, max_iter=200))])

ptrain, ptest = s[s.Year <= 2022].copy(), s[s.Year >= 2023].copy()
print(len(ptrain), "train rows |", len(ptest), "test rows")

96925 train rows | 29194 test rows


In [6]:
reg_fant, reg_goal = make_reg(), make_reg()
reg_fant.fit(ptrain[FEATS], ptrain["Fantasy"])
reg_goal.fit(ptrain[FEATS], ptrain["Goals"])

ptest["PredFantasy"] = reg_fant.predict(ptest[FEATS])
ptest["PredGoals"] = reg_goal.predict(ptest[FEATS])

print("Fantasy MAE  model:", round(mean_absolute_error(ptest["Fantasy"], ptest["PredFantasy"]), 2),
      "| baseline (last-5 avg):", round(mean_absolute_error(ptest["Fantasy"], ptest["Fantasy_L5"]), 2))
print("Goals   MAE  model:", round(mean_absolute_error(ptest["Goals"], ptest["PredGoals"]), 3),
      "| baseline (last-5 avg):", round(mean_absolute_error(ptest["Goals"], ptest["Goals_L5"]), 3))

def hit_rates(df, pred_col, actual_col):
    actual = df.loc[df.groupby("GameKey")[actual_col].idxmax()].set_index("GameKey")["PlayerId"]
    top3 = (df.sort_values(pred_col, ascending=False).groupby("GameKey").head(3)
              .groupby("GameKey")["PlayerId"].apply(set))
    top1 = (df.loc[df.groupby("GameKey")[pred_col].idxmax()].set_index("GameKey")["PlayerId"]
              .reindex(actual.index) == actual).mean()
    t3 = np.mean([actual[k] in top3[k] for k in actual.index])
    return round(top1, 3), round(t3, 3)

print("\nPick the best player in a game (top-1, top-3 hit rate)")
print("Model     :", hit_rates(ptest, "PredFantasy", "Fantasy"))
print("Last-5 avg:", hit_rates(ptest, "Fantasy_L5", "Fantasy"))
print("Random guess is about (0.02, 0.07) with ~44 players per game")

Fantasy MAE  model: 17.44 | baseline (last-5 avg): 18.3
Goals   MAE  model: 0.531 | baseline (last-5 avg): 0.517

Pick the best player in a game (top-1, top-3 hit rate)
Model     : (np.float64(0.202), np.float64(0.467))
Last-5 avg: (np.float64(0.202), np.float64(0.444))
Random guess is about (0.02, 0.07) with ~44 players per game


In [7]:
reg_fant.fit(s[FEATS], s["Fantasy"])
reg_goal.fit(s[FEATS], s["Goals"])

# latest form for every player (includes their most recent game)
s_sorted = s.sort_values("Date")
gp = s_sorted.groupby("PlayerId")
snap = gp.agg(PlayerName=("PlayerName", "last"), Team=("Team", "last"), LastDate=("Date", "last"),
              Position=("Position", "last"), Height=("Height", "last"), Weight=("Weight", "last"),
              Dob=("Dob", "last"), PrevGames=("Fantasy", "size"))
for c in ["Fantasy", "Goals", "Disposals", "PctPlayed"]:
    snap[f"{c}_L5"] = gp[c].apply(lambda x: x.tail(5).mean())
for c in ["Fantasy", "Goals", "Disposals"]:
    snap[f"{c}_L20"] = gp[c].apply(lambda x: x.tail(20).mean())
    snap[f"{c}_Career"] = gp[c].mean()
snap["Age"] = (s["Date"].max() - snap["Dob"]).dt.days / 365.25
snap = snap.reset_index()

joblib.dump({"fantasy": reg_fant, "goals": reg_goal, "snap": snap, "feats": FEATS},
            "models/player_model.joblib")

def top_players(team, is_home=True, n=5, by="fantasy"):
    t = snap[snap["Team"] == team]
    if t.empty: raise ValueError("Unknown team")
    pool = t[t["LastDate"] >= t["LastDate"].max() - pd.Timedelta(days=35)].copy()  # recently played
    pool["IsHome"] = int(is_home)
    pool["PredFantasy"] = reg_fant.predict(pool[FEATS])
    pool["PredGoals"] = reg_goal.predict(pool[FEATS])
    col = "PredFantasy" if by == "fantasy" else "PredGoals"
    return pool.nlargest(n, col)[["PlayerName", "Position", "PredFantasy", "PredGoals"]].round(2)

print(top_players("Collingwood", is_home=True, n=5))
print(top_players("Carlton", is_home=False, n=5, by="goals"))

            PlayerName            Position  PredFantasy  PredGoals
1480       Nick Daicos  Defender, Midfield       101.32       0.58
1124     Darcy Cameron             Forward        97.88       0.45
1036       Josh Daicos   Midfield, Forward        90.37       0.29
577         Jack Crisp            Defender        84.09       0.34
202   Scott Pendlebury            Midfield        75.91       0.22
          PlayerName            Position  PredFantasy  PredGoals
1072     Harry McKay             Forward        77.17       2.11
946   Charlie Curnow             Forward        65.84       1.80
1385   Francis Evans             Forward        59.51       1.17
1644     Ashton Moir             Forward        44.38       1.02
767     Zac Williams  Defender, Midfield        64.46       0.81
